<a href="https://colab.research.google.com/github/marinaalfarias/ia940/blob/main/Aula13_Panorama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 13 — Panorama: o que mais o aprendizado de máquina faz com áudio

**IA/EG940 · 2s2026**

Até aqui, o curso tratou sobretudo de **classificação**, e todos os embeddings partiram de MFCCs ou de espectrogramas. Hoje, um panorama: **separação de fontes** (Ex. 1), um embedding aprendido **direto da forma de onda** (Ex. 2) e outras aplicações do estado da arte (Ex. 3), mostradas pelo professor.

Em vários exercícios aparece **Previsão**. Faça uma cópia do notebook (*Arquivo → Salvar uma cópia no Drive*) e escreva sua resposta na célula livre *antes* de rodar a célula correspondente (ou antes da demonstração). Errar a previsão faz parte; o que importa é explicar depois por que errou.

In [ ]:
# Configuração (rode uma vez)
import numpy as np
import matplotlib.pyplot as plt
import librosa
from IPython.display import Audio, display
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

SR = 22050
rng = np.random.default_rng(10)      # mesma semente da Aula 10
plt.rcParams['figure.figsize'] = (9, 3.5)

def ouvir(x, sr=SR):
    display(Audio(x, rate=sr))

In [ ]:
import keras
from keras import layers
import tensorflow as tf
tf.get_logger().setLevel('ERROR')                  # esconde avisos internos do TensorFlow
print('Keras', keras.__version__)

### Os sons da Aula 10

Exatamente a mesma base sintética da Aula 10: 40 sons **tonais**, 40 **percussivos** e 40 **pios**, com ruído de fundo. Cada som vira o vetor de 12 MFCCs médios. Rode e siga em frente.

In [ ]:
def ganho_formantes(f, formantes):
    g = np.full_like(np.asarray(f, float), 1e-3)
    for fc, bw in formantes:
        g = g + 1 / (1 + ((f - fc) / (bw / 2))**2)
    return g

def envoltoria(n, ataque=0.05, decaimento=2.0):
    t = np.arange(n) / SR
    return np.minimum(t / ataque, 1) * np.exp(-t / decaimento)

def normaliza(x):
    return 0.5 * x / np.max(np.abs(x))

def tom(f0, formantes, dur=1.0, ataque=0.05, decaimento=2.0):
    n = int(dur * SR); t = np.arange(n) / SR; x = np.zeros(n)
    for k in range(1, int(SR / 2 / f0)):
        x += ganho_formantes(k * f0, formantes) * np.sin(2*np.pi*k*f0*t + rng.uniform(0, 2*np.pi))
    return normaliza(x * envoltoria(n, ataque, decaimento))

def ruido(formantes, dur=1.0, ataque=0.05, decaimento=2.0):
    n = int(dur * SR)
    X = np.fft.rfft(rng.standard_normal(n)); f = np.fft.rfftfreq(n, 1/SR)
    x = np.fft.irfft(X * ganho_formantes(f, formantes), n)
    return normaliza(x * envoltoria(n, ataque, decaimento))

def pios(dur=1.0):
    n = int(dur * SR); x = np.zeros(n)
    n_pios = rng.integers(2, 6); fb = rng.uniform(1500, 4500); L = int(rng.uniform(0.06, 0.18) * SR)
    for i in range(n_pios):
        ini = int(i * n / n_pios + rng.uniform(0, 0.05) * SR)
        if ini + L > n: break
        t = np.arange(L) / SR
        f = fb + rng.uniform(-1500, 1500) * t / t[-1]
        x[ini:ini+L] += np.sin(2*np.pi*np.cumsum(f)/SR) * np.hanning(L)
    return normaliza(x)

def com_ruido(x, snr_db):
    r = rng.standard_normal(len(x))
    r *= np.sqrt(np.mean(x**2) / np.mean(r**2) / 10**(snr_db/10))
    return x + r

E_base = [(500, 150), (1500, 200), (2500, 250)]
NOMES = ['tonal', 'percussivo', 'pios']
CORES = np.array(['#1f77b4', '#ff7f0e', '#2ca02c'])

def exemplo(classe):
    if classe == 0:
        e = rng.uniform(0.6, 1.6)
        x = tom(rng.uniform(110, 660), [(fc*e, bw*e) for fc, bw in E_base],
                ataque=rng.uniform(0.03, 0.3), decaimento=rng.uniform(0.8, 3))
    elif classe == 1:
        e = rng.uniform(0.4, 3)
        x = ruido([(fc*e, bw*e) for fc, bw in E_base], ataque=0.002, decaimento=rng.uniform(0.03, 0.25))
    else:
        x = pios()
    return com_ruido(x, rng.uniform(0, 20))

def vetor_mfcc(x):
    return librosa.feature.mfcc(y=x, sr=SR, n_mfcc=13)[1:].mean(axis=1)

# Refaz os mesmos sorteios da Aula 10 (sons A, B e C do Ex. 1, depois a base)
for E in ([(500, 150), (1500, 200), (2500, 250)], [(250, 100), (4000, 400), (7000, 600)]):
    com_ruido(tom(220, E), 40)
ruido([(500, 150), (1500, 200), (2500, 250)])

y = np.repeat([0, 1, 2], 40)
sons = [exemplo(c) for c in y]
MF = np.array([vetor_mfcc(x) for x in sons])
print('Base:', MF.shape)

### Ferramentas de visualização

Esta célula define três funções de desenho: um gráfico em que **clicar num ponto toca o som** correspondente, um desenho de redes densas e um diagrama de blocos. Não é preciso ler o código.

In [ ]:
import base64, io, uuid, json
from scipy.io import wavfile
from scipy.signal import resample_poly
from IPython.display import HTML

def _wav_b64(x):
    x = resample_poly(x, 320, 441)                       # 22050 -> 16000 Hz, para o arquivo ficar menor
    x = np.int16(0.8 * x / (np.max(np.abs(x)) + 1e-12) * 32767)
    b = io.BytesIO(); wavfile.write(b, 16000, x)
    return base64.b64encode(b.getvalue()).decode()

def grafico_ouvivel(paineis, sons, rotulos, nomes=NOMES, cores=CORES, lado=320):
    # paineis: lista de (título, E), com E de forma (n_sons, 2). Clique num ponto para ouvir.
    uid = 'g' + uuid.uuid4().hex[:8]
    svgs = []
    for titulo, E in paineis:
        E = np.asarray(E, float)
        lo, hi = np.percentile(E, 1, axis=0), np.percentile(E, 99, axis=0)   # ignora pontos muito afastados
        m = 0.08 * (hi - lo + 1e-12)
        px = 20 + (lado - 40) * (E[:, 0] - lo[0] + m[0]) / (hi[0] - lo[0] + 2 * m[0])
        py = 30 + (lado - 50) * (1 - (E[:, 1] - lo[1] + m[1]) / (hi[1] - lo[1] + 2 * m[1]))
        px, py = np.clip(px, 6, lado - 6), np.clip(py, 26, lado - 6)        # os afastados ficam na borda
        pts = ''.join(
            f'<circle cx="{px[i]:.1f}" cy="{py[i]:.1f}" r="4" fill="{cores[rotulos[i]]}" fill-opacity="0.8" '
            f'data-i="{i}" style="cursor:pointer"><title>som {i} ({nomes[rotulos[i]]})</title></circle>'
            for i in range(len(E)))
        svgs.append(f'<svg width="{lado}" height="{lado}" style="border:1px solid #ccc;background:#fff">'
                    f'<text x="{lado/2}" y="18" text-anchor="middle" font-size="13" font-family="sans-serif">{titulo}</text>'
                    f'{pts}</svg>')
    legenda = ' '.join(f'<span style="color:{cores[c]}">&#9679;</span> {nomes[c]}' for c in range(len(nomes)))
    audios = json.dumps([_wav_b64(x) for x in sons])
    script = '''
<script>
(function(){
  const A = __AUDIOS__, lab = __LAB__, nomes = __NOMES__;
  const raiz = document.getElementById('__UID__');
  const info = document.getElementById('__UID___info');
  const pts = raiz.querySelectorAll('circle');
  let hist = [];
  pts.forEach(c => c.addEventListener('click', () => {
    const i = +c.dataset.i;
    raiz.querySelectorAll('circle[data-i="' + i + '"]').forEach(q => q.parentNode.appendChild(q));
    new Audio('data:audio/wav;base64,' + A[i]).play();
    hist = [i].concat(hist.filter(j => j !== i)).slice(0, 2);
    pts.forEach(p => {
      const j = +p.dataset.i;
      p.setAttribute('r', hist.includes(j) ? 7 : 4);
      p.setAttribute('stroke', j === hist[0] ? 'black' : (j === hist[1] ? 'gray' : 'none'));
      p.setAttribute('stroke-width', 2);
    });
    info.textContent = 'Agora: som ' + i + ' (' + nomes[lab[i]] + ')' +
      (hist.length > 1 ? '   |   antes: som ' + hist[1] + ' (' + nomes[lab[hist[1]]] + ')' : '');
  }));
})();
</script>'''
    script = (script.replace('__AUDIOS__', audios).replace('__LAB__', json.dumps([int(r) for r in rotulos]))
                    .replace('__NOMES__', json.dumps(list(nomes))).replace('__UID__', uid))
    html = (f'<div id="{uid}" style="font-family:sans-serif;font-size:13px">'
            f'<div style="display:flex;gap:12px;flex-wrap:wrap">{"".join(svgs)}</div>'
            f'<div style="margin-top:6px">{legenda}</div>'
            f'<div id="{uid}_info" style="margin-top:4px;color:#444">Clique num ponto para ouvir o som. '
            f'O mesmo som fica destacado em todos os painéis.</div></div>' + script)
    display(HTML(html))

def desenha_dense(tamanhos, nomes_camadas, titulo=''):
    # Desenha uma rede densa: cada neurônio de uma camada liga-se a todos da camada seguinte.
    fig, ax = plt.subplots(figsize=(1.8 * len(tamanhos) + 1, 4))
    pos = []
    for k, n in enumerate(tamanhos):
        ys = np.linspace(0, 1, n) if n > 1 else np.array([0.5])
        pos.append([(k, yy) for yy in ys])
    for k in range(len(tamanhos) - 1):
        for (x0, y0) in pos[k]:
            for (x1, y1) in pos[k + 1]:
                ax.plot([x0, x1], [y0, y1], color='gray', lw=0.4, alpha=0.6)
    for k, camada in enumerate(pos):
        ax.scatter(*zip(*camada), s=90, c='white', edgecolors='k', zorder=3)
        ax.text(k, -0.12, nomes_camadas[k], ha='center', va='top', fontsize=9)
    ax.set_xlim(-0.6, len(tamanhos) - 0.4); ax.set_ylim(-0.45, 1.1); ax.axis('off')
    ax.set_title(titulo, fontsize=10); plt.tight_layout(); plt.show()

def desenha_blocos(blocos, titulo=''):
    # blocos: lista de (nome, forma da saída); desenha da esquerda para a direita.
    fig, ax = plt.subplots(figsize=(2.3 * len(blocos), 1.8))
    for k, (nome, forma) in enumerate(blocos):
        ax.add_patch(plt.Rectangle((k * 2.3, 0), 1.9, 1, fc='#eef3fb', ec='k'))
        ax.text(k * 2.3 + 0.95, 0.62, nome, ha='center', va='center', fontsize=9)
        ax.text(k * 2.3 + 0.95, 0.25, forma, ha='center', va='center', fontsize=8, color='#555')
        if k < len(blocos) - 1:
            ax.annotate('', xy=(k * 2.3 + 2.3, 0.5), xytext=(k * 2.3 + 1.9, 0.5),
                        arrowprops=dict(arrowstyle='->'))
    ax.set_xlim(-0.2, 2.3 * len(blocos)); ax.set_ylim(-0.2, 1.3); ax.axis('off')
    ax.set_title(titulo, fontsize=10); plt.tight_layout(); plt.show()

---
## Exercício 1 — Separação de fontes: uma máscara na STFT

Até aqui, as redes resumiram cada som num vetor. Agora a saída da rede tem **o mesmo tamanho da entrada**: para separar duas fontes misturadas, a rede estima, para cada célula tempo-frequência da STFT, quanto daquela célula pertence à fonte que queremos. Esse mapa de números entre 0 e 1 é uma **máscara**.

Antes de ver a rede, fazemos a separação com uma máscara **ideal**, calculada a partir das fontes originais (que, num problema real, não conhecemos).

In [ ]:
from scipy.signal import stft, istft

def espectro(x):
    f, t, X = stft(x, SR, nperseg=1024, noverlap=768)
    return f, t, X

def ressintetiza(X):
    return istft(X, SR, nperseg=1024, noverlap=768)[1]

def sdr(ref, est):
    # quanto a estimativa se parece com a referência, em dB (quanto maior, melhor)
    n = min(len(ref), len(est))
    return 10 * np.log10(np.sum(ref[:n]**2) / np.sum((ref[:n] - est[:n])**2))

tom_ = tom(220, E_base, dur=1.0, ataque=0.05, decaimento=3)
pio_ = pios()
tom_, pio_ = tom_ / np.std(tom_), pio_ / np.std(pio_)              # mesma potência: 0 dB entre as fontes
mistura = tom_ + pio_

f, t, X = espectro(mistura)
_, _, S_tom = espectro(tom_)
_, _, S_pio = espectro(pio_)

fig, eixos = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, (nome, S) in zip(eixos, [('tom', S_tom), ('pio', S_pio), ('mistura', X)]):
    ax.pcolormesh(t, f / 1000, 20 * np.log10(np.abs(S) + 1e-6), vmin=-80, vmax=0, cmap='magma', shading='auto')
    ax.set_title(nome); ax.set_xlabel('tempo (s)'); ax.set_ylim(0, 8)
eixos[0].set_ylabel('frequência (kHz)'); plt.tight_layout(); plt.show()
print(f'STFT da mistura: {X.shape[0]} frequências × {X.shape[1]} quadros')
print('Mistura'); ouvir(mistura)

### 1a. A máscara ideal

A **máscara de razão ideal** dá, em cada célula, a fração da energia que pertence ao tom:

$$M(f, t) = \frac{|S_{\text{tom}}(f, t)|^2}{|S_{\text{tom}}(f, t)|^2 + |S_{\text{pio}}(f, t)|^2}$$

Para separar, multiplicamos a STFT da **mistura** pela máscara, célula a célula, e voltamos ao tempo com a STFT inversa. A fase usada é a da mistura.

**Previsão:** a máscara do pio é igual a quê? Como soará o tom separado?

✏️ **Sua previsão:** *(clique duas vezes para escrever, antes de rodar a próxima célula)*

In [ ]:
M = np.abs(S_tom)**2 / (np.abs(S_tom)**2 + np.abs(S_pio)**2 + 1e-12)

plt.figure(figsize=(5, 3))
plt.pcolormesh(t, f / 1000, M, cmap='gray', vmin=0, vmax=1, shading='auto')
plt.colorbar(label='M'); plt.ylim(0, 8); plt.xlabel('tempo (s)'); plt.ylabel('frequência (kHz)')
plt.title('Máscara ideal do tom (branco = tom, preto = pio)', fontsize=10); plt.tight_layout(); plt.show()

tom_sep = ressintetiza(M * X)
pio_sep = ressintetiza((1 - M) * X)
print(f'Tom separado: SDR = {sdr(tom_, tom_sep):.1f} dB'); ouvir(tom_sep)
print(f'Pio separado: SDR = {sdr(pio_, pio_sep):.1f} dB'); ouvir(pio_sep)
print(f'(Para comparação, a mistura sem separação: SDR = {sdr(tom_, mistura):.1f} dB em relação ao tom)')

### 1b. E se a máscara não for perfeita?

Uma rede nunca acerta a máscara ideal. Simulamos uma estimativa imperfeita **borrando** a máscara ideal: cada grupo escolhe um grau de borrão (em células da STFT).

**Previsão:** o que acontece com o som do tom separado à medida que a máscara fica mais borrada?

✏️ **Sua previsão:** *(clique duas vezes para escrever, antes de rodar a próxima célula)*

In [ ]:
from scipy.ndimage import gaussian_filter

MEU_BORRAO = 3                 # <- cada grupo escolhe: 0, 1, 3 ou 8

M_est = gaussian_filter(M, MEU_BORRAO) if MEU_BORRAO > 0 else M
tom_est = ressintetiza(M_est * X)
print(f'borrão {MEU_BORRAO}: SDR do tom = {sdr(tom_, tom_est):.1f} dB'); ouvir(tom_est)

### 1c. A U-Net: a rede que estima a máscara

A **U-Net** é a rede que, num separador real (o Spleeter, por exemplo), faz o papel da máscara ideal: ela recebe **só a mistura** e estima a máscara.

- **Entrada:** a magnitude da STFT da mistura, $|X(f, t)|$, como uma imagem de frequências × quadros.
- **Codificador (lado esquerdo do U):** camadas convolucionais 2D, como no YAMNet. A cada nível, a imagem é reduzida à metade em frequência e em tempo, e o número de canais (filtros) dobra. No fundo do U, a mistura foi resumida numa representação pequena e "abstrata".
- **Decodificador (lado direito):** faz o caminho inverso, ampliando a imagem até o tamanho original.
- **Conexões de salto (setas horizontais):** cada nível do codificador é copiado para o nível correspondente do decodificador. Sem elas, os detalhes finos (onde exatamente está cada harmônico) se perderiam no fundo do U.
- **Saída:** uma máscara do **mesmo tamanho** da entrada, com valores entre 0 e 1 (ativação sigmoide).
- **Uso:** exatamente o que fizemos na célula 1a: máscara × STFT da mistura, com a fase da mistura, e STFT inversa.
- **Treino:** com pares (mistura, fonte isolada) feitos somando gravações separadas. A perda compara a máscara estimada × mistura com a fonte isolada.

In [ ]:
def desenha_unet():
    fig, ax = plt.subplots(figsize=(11, 4.5))
    niveis = [('513 × 88\n1 canal', 1.0), ('256 × 44\n16', 0.8), ('128 × 22\n32', 0.6), ('64 × 11\n64', 0.4)]
    for k, (txt, h) in enumerate(niveis):
        y0 = 3.2 - k * 0.9
        ax.add_patch(plt.Rectangle((k * 0.9, y0), 0.7, h * 0.8, fc='#eef3fb', ec='k'))
        ax.text(k * 0.9 + 0.35, y0 + h * 0.4, txt, ha='center', va='center', fontsize=7)
        xr = 8.2 - k * 0.9
        ax.add_patch(plt.Rectangle((xr, y0), 0.7, h * 0.8, fc='#fdf0e6', ec='k'))
        ax.text(xr + 0.35, y0 + h * 0.4, txt.replace('1 canal', 'máscara'), ha='center', va='center', fontsize=7)
        ax.annotate('', xy=(xr, y0 + h * 0.4), xytext=(k * 0.9 + 0.7, y0 + h * 0.4),
                    arrowprops=dict(arrowstyle='->', ls='--', color='gray'))
    ax.add_patch(plt.Rectangle((3.9, 0.2), 1.1, 0.4, fc='#e8f5e9', ec='k'))
    ax.text(4.45, 0.4, 'fundo do U\n32 × 6, 128 canais', ha='center', va='center', fontsize=7)
    ax.text(0.35, 4.3, 'entrada: |STFT da mistura|', ha='center', fontsize=9)
    ax.text(8.55, 4.3, 'saída: máscara (0 a 1)', ha='center', fontsize=9)
    ax.text(1.6, 0.4, 'codificador:\nconv 2D, reduz à metade', ha='center', fontsize=8)
    ax.text(7.3, 0.4, 'decodificador:\namplia ao dobro', ha='center', fontsize=8)
    ax.text(4.45, 3.4, 'setas tracejadas: conexões de salto', ha='center', fontsize=8, color='gray')
    ax.set_xlim(-0.2, 9.2); ax.set_ylim(0, 4.6); ax.axis('off')
    ax.set_title('U-Net estimando a máscara (tamanhos ilustrativos)', fontsize=10)
    plt.tight_layout(); plt.show()

desenha_unet()

**Para discutir:** o que entra na U-Net e o que sai? Por que estimar uma máscara, e não diretamente o espectrograma da fonte? Por que a separação usa a fase da mistura? Compare o fundo do U com o gargalo do autoencoder da Aula 11: por que a U-Net precisa das conexões de salto, e o autoencoder não podia tê-las?

### 1d. Uma U-Net pequena, treinada aqui

Agora treinamos uma U-Net de verdade, pequena, para estimar a máscara do tom **só a partir da mistura**. Os dados de treino são 300 misturas de um tom e de uma sequência de pios, sorteados como na mistura acima; para cada uma, sabemos a máscara ideal, que serve de alvo. A perda é o erro quadrático médio entre a máscara estimada e a ideal.

Cada grupo treina uma versão: **com** conexões de salto ou **sem** elas (sem saltos, a rede é um autoencoder: tudo precisa passar pelo fundo do U).

**Dica:** o treino leva alguns minutos. Rode a célula de treino e acompanhe a explicação enquanto ela roda.

**Previsão:** a máscara estimada pela versão do seu grupo vai se parecer com a ideal? Que SDR você espera, comparado com os do 1b?

✏️ **Sua previsão:** *(clique duas vezes para escrever, antes de rodar a próxima célula)*

In [ ]:
NPS, HOP = 512, 128                      # STFT menor que a do 1a, para a rede treinar rápido
NF, NT = 256, 168                        # recorte: 256 frequências × 168 quadros (divisíveis por 8)

def stft_pequena(x):
    return stft(x, SR, nperseg=NPS, noverlap=NPS - HOP)[2]

def exemplo_unet(a, b):
    A, B = stft_pequena(a), stft_pequena(b)
    X_ = A + B
    M_ = np.abs(A)**2 / (np.abs(A)**2 + np.abs(B)**2 + 1e-12)        # máscara ideal do tom
    return X_[:NF, :NT], M_[:NF, :NT]

def par_aleatorio():
    e = rng.uniform(0.6, 1.6)
    a = tom(rng.uniform(110, 440), [(fc*e, bw*e) for fc, bw in E_base], dur=1.0,
            ataque=0.05, decaimento=rng.uniform(1, 3))
    b = pios()
    return a / np.std(a), b / np.std(b)

N_TREINO = 300
Xs, Ms = zip(*[exemplo_unet(*par_aleatorio()) for _ in range(N_TREINO)])
log_mag = np.log(np.abs(np.array(Xs)) + 1e-3)[..., None]            # entrada: log da magnitude da mistura
mu_u, dp_u = log_mag.mean(), log_mag.std()
entrada_unet = ((log_mag - mu_u) / dp_u).astype('float32')
alvo_unet = np.array(Ms)[..., None].astype('float32')
print('entrada:', entrada_unet.shape, '  alvo (máscara ideal):', alvo_unet.shape)

In [ ]:
def unet(saltos=True):
    keras.utils.set_random_seed(0)
    ent = keras.Input(shape=(NF, NT, 1))                                                   # 256 × 168 × 1
    e1 = layers.Conv2D(8, 3, padding='same', activation='relu')(ent)                       # 256 × 168 × 8
    e2 = layers.Conv2D(16, 3, strides=2, padding='same', activation='relu')(e1)            # 128 × 84 × 16
    e3 = layers.Conv2D(32, 3, strides=2, padding='same', activation='relu')(e2)            #  64 × 42 × 32
    fundo = layers.Conv2D(64, 3, strides=2, padding='same', activation='relu')(e3)         #  32 × 21 × 64
    d3 = layers.Conv2DTranspose(32, 3, strides=2, padding='same', activation='relu')(fundo) #  64 × 42
    if saltos: d3 = layers.Concatenate()([d3, e3])                                         # conexão de salto
    d2 = layers.Conv2DTranspose(16, 3, strides=2, padding='same', activation='relu')(d3)   # 128 × 84
    if saltos: d2 = layers.Concatenate()([d2, e2])
    d1 = layers.Conv2DTranspose(8, 3, strides=2, padding='same', activation='relu')(d2)    # 256 × 168
    if saltos: d1 = layers.Concatenate()([d1, e1])
    mascara = layers.Conv2D(1, 1, activation='sigmoid')(d1)                                # 256 × 168 × 1, entre 0 e 1
    m = keras.Model(ent, mascara)
    m.compile(optimizer=keras.optimizers.Adam(0.003), loss='mse')
    return m

MEUS_SALTOS = True             # <- metade dos grupos usa True, a outra metade False

rede_u = unet(MEUS_SALTOS)
rede_u.summary()
hist_u = rede_u.fit(entrada_unet, alvo_unet, epochs=15, batch_size=16, validation_split=0.1, verbose=0)
plt.plot(hist_u.history['loss'], label='treino'); plt.plot(hist_u.history['val_loss'], label='validação')
plt.xlabel('época'); plt.ylabel('perda'); plt.legend(); plt.grid(alpha=0.3)
plt.title(f'Treino da U-Net ({"com" if MEUS_SALTOS else "sem"} conexões de salto)', fontsize=10); plt.show()

Agora usamos a U-Net treinada na **mistura do 1a**, que ela nunca viu: estimamos a máscara, aplicamos à STFT da mistura e ressintetizamos, exatamente como fizemos com a máscara ideal.

In [ ]:
X_p = stft_pequena(mistura)                                                    # STFT completa da mistura
_, _, S_tom_p = stft(tom_, SR, nperseg=NPS, noverlap=NPS - HOP)
_, _, S_pio_p = stft(pio_, SR, nperseg=NPS, noverlap=NPS - HOP)
M_ideal_p = np.abs(S_tom_p)**2 / (np.abs(S_tom_p)**2 + np.abs(S_pio_p)**2 + 1e-12)

ent_p = ((np.log(np.abs(X_p[:NF, :NT]) + 1e-3) - mu_u) / dp_u)[None, ..., None].astype('float32')
M_rede = np.asarray(rede_u(ent_p, training=False))[0, ..., 0]                 # máscara estimada (256 × 168)

M_p = np.ones(X_p.shape)                                                       # completa o que ficou fora do recorte
M_p[:NF, :NT] = M_rede
M_p[NF:, :NT] = M_rede[-1:, :]
M_p[:, NT:] = M_p[:, NT - 1:NT]

fig, eixos = plt.subplots(1, 3, figsize=(13, 3), sharey=True)
for ax, (nome, im, cm) in zip(eixos, [('entrada: |STFT da mistura| (dB)', 20 * np.log10(np.abs(X_p) + 1e-6), 'magma'),
                                      ('máscara ideal', M_ideal_p, 'gray'), ('máscara estimada pela U-Net', M_p, 'gray')]):
    ax.imshow(im[:NF], origin='lower', aspect='auto', cmap=cm, vmin=(-80 if cm == 'magma' else 0), vmax=(0 if cm == 'magma' else 1))
    ax.set_title(nome, fontsize=10); ax.set_xlabel('quadro')
eixos[0].set_ylabel('frequência (bin)'); plt.tight_layout(); plt.show()

r = lambda X_: istft(X_, SR, nperseg=NPS, noverlap=NPS - HOP)[1]
print(f'Máscara ideal:              SDR do tom = {sdr(tom_, r(M_ideal_p * X_p)):.1f} dB')
print(f'Máscara estimada (U-Net):   SDR do tom = {sdr(tom_, r(M_p * X_p)):.1f} dB'); ouvir(r(M_p * X_p))

**Para discutir:** a máscara estimada se parece com a ideal? Onde ela erra? Compare o SDR do seu grupo com o de um grupo que usou a outra versão (com ou sem saltos) e com os SDRs do 1b. O que as conexões de salto mudam?

---
## Exercício 2 — Um embedding direto da forma de onda

Até aqui, toda rede recebeu algo calculado por nós: MFCCs, um mel-espectrograma, uma STFT. Agora a rede recebe **as amostras do áudio**, e a primeira camada é uma convolução 1D diretamente sobre elas: cada filtro é um **FIR de 128 coeficientes** (8 ms a 16 kHz), com coeficientes aprendidos. A saída de cada filtro passa por uma ReLU e por uma média em blocos de 512 amostras (32 ms): é a **energia** que passa por aquele filtro em cada instante, como uma banda de um banco de filtros.

A pergunta: se ninguém disser à rede que bandas usar, que filtros ela aprende?

Usamos a base da Aula 10 (tonal, percussivo e pios), reamostrada para 16 kHz.

In [ ]:
from scipy.signal import resample_poly

SR16 = 16000
ondas = np.array([resample_poly(x, 320, 441)[:SR16] for x in sons])        # 22050 -> 16000 Hz, 1 s
ondas = (ondas / np.abs(ondas).max(axis=1, keepdims=True))[..., None].astype('float32')
print('Entrada da rede:', ondas.shape, '(sons, amostras, 1 canal)')

desenha_blocos([('forma de onda', '(16000, 1)'), ('Conv1D: 16 filtros\nFIR de 128 amostras', '(16000, 16)'),
                ('ReLU + média\nem blocos de 32 ms', '(31, 16)'), ('Conv1D: 32 filtros\nde 3 blocos', '(29, 32)'),
                ('média no tempo\n= embedding', '(32,)'), ('Dense(3)\nsoftmax', 'tonal, perc., pio')],
               'Rede sobre a forma de onda')

### 2a. Que filtros a rede aprende?

**Previsão:** depois do treino, como serão as respostas em frequência dos 16 filtros da primeira camada? Iguais entre si, passa-faixas em frequências diferentes, ou outra coisa?

✏️ **Sua previsão:** *(clique duas vezes para escrever, antes de rodar a próxima célula)*

In [ ]:
def rede_onda(semente=0, treinar_filtros=True):
    keras.utils.set_random_seed(semente)
    ent = keras.Input(shape=(SR16, 1))
    h = layers.Conv1D(16, 128, padding='same', use_bias=False, name='filtros',
                      trainable=treinar_filtros)(ent)                                  # banco de 16 FIRs
    h = layers.Activation('relu')(h)
    h = layers.AveragePooling1D(512)(h)                                               # energia por bloco de 32 ms
    h = layers.Lambda(lambda z: keras.ops.log(z + 1e-4))(h)                           # em escala log, como o log-mel
    h = layers.Conv1D(32, 3, activation='relu')(h)
    emb = layers.GlobalAveragePooling1D(name='embedding')(h)
    saida = layers.Dense(3, activation='softmax')(emb)
    m = keras.Model(ent, saida)
    m.compile(optimizer=keras.optimizers.Adam(0.003), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

a, b, ya, yb = train_test_split(ondas, y, test_size=0.3, stratify=y, random_state=0)
rede_w = rede_onda()
W0 = rede_w.get_layer('filtros').get_weights()[0][:, 0, :].copy()                  # filtros antes do treino
rede_w.fit(a, ya, epochs=60, batch_size=16, verbose=0)
W1 = rede_w.get_layer('filtros').get_weights()[0][:, 0, :]                          # filtros depois do treino
print(f'Acurácia de teste da rede sobre a forma de onda: {rede_w.evaluate(b, yb, verbose=0)[1]:.0%}')
print('(Para comparação: o kNN com MFCC médio na mesma base deu 86%, e com o YAMNet, 99%.)')

In [ ]:
def respostas(W):
    H = np.abs(np.fft.rfft(W, n=1024, axis=0))                                      # (513, 16)
    H = 20 * np.log10(H / H.max(axis=0, keepdims=True) + 1e-6)
    ordem = np.argsort(np.argmax(H, axis=0))                                         # ordena pelo pico
    return H[:, ordem]

f_w = np.fft.rfftfreq(1024, 1 / SR16) / 1000
fig, eixos = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for ax, (nome, W) in zip(eixos, [('antes do treino (aleatórios)', W0), ('depois do treino', W1)]):
    ax.imshow(respostas(W), origin='lower', aspect='auto', cmap='magma', vmin=-30, vmax=0,
              extent=[0.5, 16.5, 0, f_w[-1]])
    ax.set_title(f'Resposta em frequência dos 16 filtros, {nome}', fontsize=10); ax.set_xlabel('filtro (ordenado pelo pico)')
eixos[0].set_ylabel('frequência (kHz)'); plt.tight_layout(); plt.show()

**Para discutir:** os filtros mudaram com o treino? Em que faixas eles se concentram, e isso tem relação com os sons da base (tons até alguns kHz, pios entre 1,5 e 4,5 kHz)? Com só 84 sons de treino, o que você espera de um banco de filtros aprendido?

### 2b. Os filtros aprendidos fazem diferença?

Para saber se o treino dos filtros importou, treinamos a mesma rede com a primeira camada **congelada** (`trainable=False`): os 16 filtros ficam aleatórios, como antes do treino, e só as camadas seguintes aprendem. Rodamos as duas versões com 5 sementes.

**Previsão:** a rede com filtros aleatórios congelados vai acertar muito menos, um pouco menos ou o mesmo que a rede com filtros treinados?

✏️ **Sua previsão:** *(clique duas vezes para escrever, antes de rodar a próxima célula)*

In [ ]:
acc = {True: [], False: []}
for semente in range(5):
    for treinar in [True, False]:
        m = rede_onda(semente, treinar_filtros=treinar)
        m.fit(a, ya, epochs=60, batch_size=16, verbose=0)
        acc[treinar].append(m.evaluate(b, yb, verbose=0)[1])
for treinar, nome in [(True, 'filtros treinados'), (False, 'filtros aleatórios, congelados')]:
    print(f'{nome:32s}', '  '.join(f'{v:.0%}' for v in acc[treinar]), f'  média = {np.mean(acc[treinar]):.0%}')

In [ ]:
for treinar in [True, False]:
    m = rede_onda(0, treinar_filtros=treinar)
    h = m.fit(a, ya, epochs=60, batch_size=16, verbose=0)
    print(f'filtros treinados={treinar}:  treino {h.history["accuracy"][-1]:.0%}   teste {m.evaluate(b, yb, verbose=0)[1]:.0%}')

**Para discutir:** o que esse resultado diz sobre a dificuldade da nossa base? De onde vem a informação que permite classificar, se não dos filtros? Se a base fosse de vozes ou de instrumentos, você esperaria a mesma conclusão?

### 2c. A mesma camada, aprendida com 960 horas de fala (demonstração)

O professor vai mostrar a primeira camada do **wav2vec 2.0**: 512 filtros de 10 amostras sobre a forma de onda a 16 kHz, treinados com 960 horas de fala **sem nenhum rótulo** (a rede aprende a adivinhar trechos escondidos do áudio). Depois, o embedding dele (768 números) entra no mesmo kNN da Aula 12.

**Previsão:** os filtros do wav2vec 2.0 vão cobrir todas as frequências por igual, ou se concentrar em alguma faixa? O kNN com o embedding dele, que só ouviu fala, vai se sair melhor ou pior que com o MFCC (86%) e com o YAMNet (99%)?

✏️ **Sua previsão:** *(clique duas vezes para escrever, antes de rodar a próxima célula)*

---
## Exercício 3 — Panorama: outras aplicações

As demonstrações deste exercício rodam no computador do professor, porque usam modelos grandes. Antes de cada uma, registre sua previsão aqui.

### 3a. Reconhecimento de fala (Whisper)

O Whisper recebe áudio e devolve **texto**: sua saída é uma sequência de tokens de texto, um pedaço de palavra por vez. Foi treinado com 680 mil horas de áudio com transcrições.

**Previsão:** o que o Whisper vai "transcrever" quando receber um som que não é fala, como um pio ou um tom da nossa base?

✏️ **Sua previsão:** *(clique duas vezes para escrever, antes de rodar a próxima célula)*

### 3b. Áudio e texto no mesmo espaço (CLAP)

O CLAP tem dois codificadores, um para áudio e um para texto, treinados juntos com centenas de milhares de pares (som, descrição), com a regra: "o embedding de um som deve ficar perto do embedding da sua descrição, e longe das outras". Depois do treino, dá para classificar um som **escrevendo as classes em palavras**, sem nenhum exemplo de treino: basta ver de qual descrição o som fica mais perto.

**Previsão:** descrevendo nossas três classes como "a sustained musical tone", "a short noise burst" e "birds chirping", que acurácia o CLAP terá na base, sem ver nenhum exemplo? Melhor ou pior que o kNN com o MFCC (86%)?

✏️ **Sua previsão:** *(clique duas vezes para escrever, antes de rodar a próxima célula)*

### 3c. Codec neural e tokens (EnCodec)

O EnCodec é um autoencoder sobre a forma de onda com um gargalo **quantizado**: o embedding de cada trecho de 13 ms é trocado pelo índice do vetor mais próximo num dicionário de 1024 vetores. O áudio vira uma sequência de **números inteiros**, os **tokens**, e o decodificador reconstrói o som a partir deles. Quanto mais dicionários em sequência, mais tokens por segundo e melhor a qualidade. Modelos de geração de música, como o MusicGen, geram exatamente esses tokens, um após o outro, como um modelo de linguagem gera palavras.

**Previsão:** a 1,5 kbps (cerca de 250 vezes menos que o áudio original), como soará a música reconstruída?

✏️ **Sua previsão:** *(clique duas vezes para escrever, antes de rodar a próxima célula)*